# Hosted teleop TwistStamped latency

End-to-end transport latency for the keyboard → Cloudflare WebRTC → robot path,
computed offline from a `HostedTeleopRecorder` SQLite recording.

- **Source stamp** (`o.data.ts`): set in the browser when the key/state was sampled, encoded into `TwistStamped.header.stamp`.
- **Receive stamp** (`o.tags["recv_ts"]`): wall-clock at the robot when the Recorder's input callback fired.

`latency = recv_ts − msg.ts` covers send → WebRTC SFU → decode → republish → record callback.
Assumes browser and robot clocks are roughly NTP-synced; if they aren't, the histogram shape is still meaningful (jitter), only the absolute offset is off.

For a live console summary while running, use `TeleopBenchmarkModule` (`dimos run teleop-hosted-go2 teleop-benchmark`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from dimos.constants import DIMOS_PROJECT_ROOT
from dimos.memory2.store.sqlite import SqliteStore

# Set this to the recording you want to analyse.
RECORDING_PATH = (
    DIMOS_PROJECT_ROOT / "data/hosted_teleop/recordings/recording_hosted_20260519_153433.db"
)

if not RECORDING_PATH.exists():
    raise FileNotFoundError(
        f"No recording at {RECORDING_PATH} — run a session with the "
        "`hosted-teleop-recorder` module first."
    )


store = SqliteStore(path=str(RECORDING_PATH))
try:
    obs = {}
    summaries = {}
    for name, stream in store.streams.items():
        items = stream.to_list()
        for o in items:
            _ = o.data  # eagerly decode while the connection is open
        obs[name] = items
        summaries[name] = stream.summary()
finally:
    store.stop()

print(f"recording: {RECORDING_PATH.name}")
for name, summary in summaries.items():
    print(f"  {summary}")

## Latency: source → recorder

Histogram (log y) with p50/p95/p99 markers. Negative values = browser clock ahead of robot clock; large positive tail = network/process stalls.

In [ ]:
# Tukey IQR fence — drop samples outside [Q1 - K*IQR, Q3 + K*IQR].
# Transport latency = recv_ts (wall-clock at robot when message arrived) − msg.ts
# (source stamp set by browser). recv_ts is stashed in obs.tags by the Recorder;
# older recordings without recv_ts will fall back to a 0ms-latency notice.

OUTLIER_K = 1.5

twist = obs.get("cmd_vel_stamped", [])
if not twist:
    print("no cmd_vel_stamped stream")
else:
    src = np.array([o.data.ts for o in twist])
    recv = np.array([o.tags.get("recv_ts", o.ts) for o in twist])
    lat_ms_all = (recv - src) * 1000.0

    if np.allclose(lat_ms_all, 0):
        print(
            "All latencies are ~0ms — recording predates recv_ts support "
            "(re-record with the updated Recorder to enable latency analysis)."
        )
        print(f"Total messages: {len(twist)}")
        print(f"Time span: {src[-1] - src[0]:.1f}s")
    else:
        q1, q3 = np.percentile(lat_ms_all, [25, 75])
        iqr = q3 - q1
        lo, hi = q1 - OUTLIER_K * iqr, q3 + OUTLIER_K * iqr
        mask = (lat_ms_all >= lo) & (lat_ms_all <= hi)
        lat_ms = lat_ms_all[mask]
        recv_inlier = recv[mask]
        dropped = len(lat_ms_all) - len(lat_ms)
        print(f"dropped {dropped}/{len(lat_ms_all)} outliers outside [{lo:.1f}, {hi:.1f}] ms")

        p50, p95, p99 = np.percentile(lat_ms, [50, 95, 99])
        print(
            f"n={len(lat_ms)}  min={lat_ms.min():.1f}  p50={p50:.1f}  p95={p95:.1f}  "
            f"p99={p99:.1f}  max={lat_ms.max():.1f}  mean={lat_ms.mean():.1f} ms"
        )

        fig, ax = plt.subplots(figsize=(11, 3.2))
        ax.hist(lat_ms, bins=80, log=True, color="steelblue", edgecolor="none")
        for p, c, lbl in [(p50, "black", "p50"), (p95, "orange", "p95"), (p99, "red", "p99")]:
            ax.axvline(p, color=c, linestyle="--", lw=0.8, label=f"{lbl}={p:.1f}ms")
        ax.set_title(
            f"cmd_vel_stamped transport latency (browser → robot), n={len(lat_ms)} ({dropped} outliers dropped)"
        )
        ax.set_xlabel("latency (ms)")
        ax.set_ylabel("count (log)")
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

## Latency over time

Drift or burst stalls show up here. Steady horizontal band = stable transport; ramps = clock drift; spikes = network jitter / SFU / GC.

In [ ]:
if twist:
    if np.allclose(lat_ms_all, 0):
        print("Skipping latency-over-time plot (recording predates recv_ts support)")
    else:
        rel = recv_inlier - recv_inlier[0]
        fig, ax = plt.subplots(figsize=(11, 3.2))
        ax.plot(rel, lat_ms, lw=0.5, color="steelblue")
        ax.axhline(p50, color="black", ls="--", lw=0.8, label=f"p50={p50:.1f}")
        ax.axhline(p95, color="orange", ls="--", lw=0.8, label=f"p95={p95:.1f}")
        ax.set_xlabel("time since first sample (s)")
        ax.set_ylabel("latency (ms)")
        ax.set_title("latency over time (outliers removed)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## Inter-arrival jitter

Source-side cadence (browser sendInterval) vs recorder-side cadence. If recorder intervals are much wider than source intervals, the transport is bunching/dropping.

In [ ]:
if twist:
    src_iv = np.diff(np.sort(src)) * 1000.0
    recv_iv = np.diff(np.sort(recv)) * 1000.0
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
    for ax, iv, lbl in [
        (axes[0], src_iv, "source (browser)"),
        (axes[1], recv_iv, "recorder (robot)"),
    ]:
        rate = 1000.0 / np.median(iv)
        p50, p95 = np.percentile(iv, [50, 95])
        ax.hist(iv, bins=80, log=True, color="steelblue", edgecolor="none")
        ax.axvline(p50, color="black", ls="--", lw=0.8, label=f"p50={p50:.1f}")
        ax.axvline(p95, color="orange", ls="--", lw=0.8, label=f"p95={p95:.1f}")
        ax.set_title(f"{lbl}: ~{rate:.1f} Hz")
        ax.set_xlabel("interval (ms)")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("count (log)")
    plt.tight_layout()
    plt.show()